<a href="https://colab.research.google.com/github/Zainaafreen/jio-careops-analyzer/blob/main/Jio_CareOps_Student_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Jio CareOps Network Issue Analyzer | Student Starter**

---

## Project Summary
**Problem:** Jio's Customer Care and Network Operations teams keep complaints, network incidents, recharge failures and service requests in separate files, so it is hard to see which cases need attention first.

**Solution:** A Python/Pandas workflow in Google Colab that loads a CSV, an Excel file and a JSON file, cleans them, joins them without duplicating complaints, and classifies each complaint as P0–P3 using four risk signals (network, recharge refund, service follow-up, repeat contact).

**Results:** 4,901 unique complaints analysed. 584 are P0 and 1,888 are P1 (50.4% of cases). Outputs: jio_complaint_priority_report.csv, jio_careops_operations_summary.xlsx, and a filter_cases search tool.

**Tools:** Python, Pandas, Google Colab, CSV/Excel/JSON.

**GitHub repo:** <https://github.com/Zainaafreen/jio-careops-analyzer>

## **Your Engineering Handover**

You have joined the Jio Customer Care and Network Operations team as a **Junior Python Developer**. The CareOps lead spends hours switching between customer complaint records, network incident reports, recharge failure logs, and service requests to understand which cases need attention. You have been asked to build a reproducible **customer complaint and network issue analysis workflow** in this notebook.

Your deliverable is **not an app or dashboard**. It is a trustworthy set of cleaned data tables, a complaint-level investigation report, and a concise evidence-based handover to Customer Care and Network Operations.

**How to work:** Make your own copy in Google Drive (`File → Save a copy in Drive`), upload the three provided files, complete the ten milestones in order, and run the assessment cell at the end. Write your own implementation in the code cells. Keep every required named output available for the next task and for assessment.

**Input files (only three):** `jio_complaints_master.csv` · `jio_incident_histories.xlsx` · `jio_service_requests.json`  
**Data:** Fictional training records; not actual Jio customer or network operations records.

> **Important:** The assessment at the end checks key output contracts and consistency. Passing it is not proof that every business calculation or written explanation is correct; reviewers may also inspect selected notebooks.

### **Important Note: Guidelines for Using AI Tools**



You are allowed to use AI tools such as ChatGPT to understand concepts, debug errors, and get assistance with **specific code snippets or individual problems**.

However, **using AI to generate the complete project solution in one go is not permitted**.

You are expected to develop the solution milestone by milestone, apply your understanding of Python and Pandas, and independently verify your code and outputs.

**Remember:** The goal of this project is not just to complete the tasks but to develop practical problem-solving, debugging, and data analysis skills that are expected in real-world development roles.

## **Workspace setup · Data intake**
Run the setup cell and upload the **three input files** when requested. Files stay in your current Colab session; if the runtime restarts, upload them again. Keep original filenames unchanged.

[Link to Dataset](https://drive.google.com/file/d/1hgqvHzcqbEDG5e_sjSXjPljwV2_u7noh/view?usp=sharing)

In [2]:
from pathlib import Path
from IPython.display import display
import pandas as pd
import json
import zipfile
DATA_DIR = Path('/content') if Path('/content').is_dir() else Path.cwd()
REQUIRED_FILES = ['complaints.csv', 'network_issue_reports.xlsx', 'careops_reference.json']
ARCHIVE = 'jio_careops_project_datasets.zip'
if any(not (DATA_DIR/name).exists() for name in REQUIRED_FILES) and (DATA_DIR/ARCHIVE).is_file():
 with zipfile.ZipFile(DATA_DIR/ARCHIVE) as z:
  names=z.namelist()
  if sorted(names)!=sorted(REQUIRED_FILES):
   raise ValueError(f'Dataset ZIP must contain exactly the three expected files: {REQUIRED_FILES}; found {names}')
  z.extractall(DATA_DIR)
if any(not (DATA_DIR/name).exists() for name in REQUIRED_FILES):
 try:
  from google.colab import files
  print('Upload jio_careops_project_datasets.zip (containing CSV, XLSX and JSON).')
  files.upload()
 except ImportError:
  print('Place the dataset ZIP or three extracted files in',DATA_DIR)
 if (DATA_DIR/ARCHIVE).is_file():
  with zipfile.ZipFile(DATA_DIR/ARCHIVE) as z:
   if sorted(z.namelist())!=sorted(REQUIRED_FILES):
    raise ValueError('Unexpected files in dataset archive')
   z.extractall(DATA_DIR)
missing=[name for name in REQUIRED_FILES if not (DATA_DIR/name).exists()]
if missing: raise FileNotFoundError(f'Missing dataset files: {missing}')
print('Verified the three required Jio sources:', ', '.join(REQUIRED_FILES))

Verified the three required Jio sources: complaints.csv, network_issue_reports.xlsx, careops_reference.json


## Milestone 01 · Establish the complaint baseline

**From the team:** The CareOps lead wants a transparent complaint intake audit before any triage recommendations are trusted.

**Your responsibility:** Load complaints.csv; inspect volume, unique complaint and customer IDs, missing values, and leading raw categories.

**Handover evidence:** Keep `complaints_df`; display shape, sample, duplicates and missing-value counts.

In [3]:
# Milestone 01 — Establish the complaint baseline
# Task 1: Load complaints.csv; inspect volume, unique complaint and customer IDs, missing values, and leading raw categories.
# Task 2: Produce this required evidence: Keep `complaints_df`; display shape, sample, duplicates and missing-value counts.
# Implement your code below and display the requested outputs.

complaints_df = pd.read_csv(DATA_DIR / 'complaints.csv')
print(complaints_df.shape)
display(complaints_df.head())
print("Rows and columns:", complaints_df.shape)
print("Unique complaint IDs:", complaints_df['complaint_id'].nunique())
print("Unique customers:", complaints_df['customer_id'].nunique())
print("Completely identical rows:", complaints_df.duplicated().sum())
print("Complaint IDs that appear more than once:", complaints_df['complaint_id'].duplicated().sum())
print("\nMissing values per column:")
display(complaints_df.isna().sum().to_frame('missing_count').query('missing_count > 0'))

print("\nMost common raw issue categories:")
display(complaints_df['issue_category'].value_counts(dropna=False).head(10))

print("\nRaw complaint statuses:")
display(complaints_df['complaint_status'].value_counts(dropna=False))

(5000, 15)


,complaint_id,customer_id,cell_site_id,complaint_date,complaint_time,city,circle,complaint_channel,issue_category,issue_description,complaint_status,sentiment_tag,network_type,device_type,amount_impacted
0,CMP000001,CUST00246,SITE0878,2026-05-04,17:04,Jaipur,Rajasthan,MyJio App,NaN,SIM activation or verification is pending afte...,Open,Urgent,VoLTE,JioPhone,799.0
1,CMP000002,CUST00793,SITE0461,2026-02-05,19:16,Ahmedabad,Gujarat,Retail Store,NaN,SIM activation or verification is pending afte...,NaN,Urgent,VoLTE,iOS,399.0
2,CMP000003,CUST00931,SITE0550,2026-02-09,20:46,Hyderabad,Andhra Pradesh & Telangana,Call Center,NaN,Fiber connection outage reported for apartment...,In Progress,Angry,5G,JioPhone,199.0
3,CMP000004,CUST00043,SITE0443,2026-02-18,13:45,Ahmedabad,Gujarat,Email,internet not working,Customer says calls drop during office hours.,Open,Frustrated,VoLTE,iOS,399.0
4,CMP000005,CUST00723,SITE0214,2026-02-01,23:28,Chennai,Tamil Nadu,Social Media,payment deducted recharge not done,Fiber connection outage reported for apartment...,closeD,Urgent,Fiber,JioPhone,399.0


Rows and columns: (5000, 15)
Unique complaint IDs: 4901
Unique customers: 1218
Completely identical rows: 0
Complaint IDs that appear more than once: 99

Missing values per column:


,missing_count
complaint_date,6
issue_category,290
complaint_status,459
amount_impacted,429



Most common raw issue categories:


,count
issue_category,
network down,416
recharge failed,356
SLOW DATA,326
No Signal,321
call drop,312
internet not working,299
NaN,290
network outage,267
service request pending,265



Raw complaint statuses:


,count
complaint_status,
closeD,477
pending review,470
NaN,459
Escalated,454
Pending,453
Resolved,452
Closed,451
in-progress,449
Open,442


## Milestone 02 · Onboard the XLSX network source and JSON reference collections

**From the team:** Network engineering provides Excel incident logs, while customer, site, recharge, and service-request histories are organized as distinct collections inside a single JSON reference file.

**Your responsibility:** Read the workbook and JSON; materialize five independent source DataFrames and policy notes. Do not merge repeated recharge or request records prematurely.

**Required evidence:** `customers_df`, `cell_sites_df`, `network_df`, `recharge_df`, `requests_df`, `operations_notes`, and source row counts.

In [4]:
# Milestone 02 — Onboard network and customer sources
# Task 1: Parse careops_reference.json and inspect its named collections and operational notes.
# Task 2: Build customers_df and cell_sites_df from their distinct JSON collections.
# Task 3: Load the XLSX source into network_df; inspect network-report identifiers and severity.
# Task 4: Convert JSON recharge_failures and service_requests collections into recharge_df and requests_df.
# Task 5: Preserve operations_notes as a string, and display source record counts and sample rows.
# Required output: customers_df, cell_sites_df, network_df, recharge_df, requests_df, operations_notes.


import json

with open(DATA_DIR / 'careops_reference.json') as f:
    reference = json.load(f)

print("Collections in the JSON:", list(reference.keys()))

customers_df  = pd.DataFrame(reference['customers'])
cell_sites_df = pd.DataFrame(reference['cell_sites'])
recharge_df   = pd.DataFrame(reference['recharge_failures'])
requests_df   = pd.DataFrame(reference['service_requests'])
operations_notes = str(reference['network_operations_notes'])

network_df = pd.read_excel(DATA_DIR / 'network_issue_reports.xlsx')

source_counts = {
    'customers_df': len(customers_df),
    'cell_sites_df': len(cell_sites_df),
    'network_df': len(network_df),
    'recharge_df': len(recharge_df),
    'requests_df': len(requests_df),
}
print("Source record counts:", source_counts)

for name, d in [('customers', customers_df), ('cell_sites', cell_sites_df),
                ('network', network_df), ('recharge', recharge_df),
                ('requests', requests_df)]:
    print("\n" + name, d.shape)
    display(d.head(3))

print("\nOperations notes (first 300 characters):")
print(operations_notes[:300])

Collections in the JSON: ['dataset_description', 'customers', 'cell_sites', 'recharge_failures', 'service_requests', 'network_operations_notes']
Source record counts: {'customers_df': 1200, 'cell_sites_df': 1000, 'network_df': 1300, 'recharge_df': 1500, 'requests_df': 1500}

customers (1200, 9)


,customer_id,customer_name,customer_segment,plan_type,home_city,circle,account_age_days,preferred_service,monthly_recharge_value
0,CUST00001,Krishna Mehta,Prepaid,4G Value,Indore,Madhya Pradesh,581,Mobile Data,1499
1,CUST00002,Vivaan Sharma,Prepaid,5G Unlimited,Delhi,Delhi NCR,905,Voice,1499
2,CUST00003,Reyansh Kulkarni,Prepaid,4G Value,Lucknow,UP East,1849,Roaming,399



cell_sites (1000, 9)


,cell_site_id,site_name,city,circle,tower_type,site_status,avg_signal_strength_dbm,capacity_utilization_pct,last_maintenance_date
0,SITE0001,DEL-NW-0001,Delhi,Delhi NCR,Indoor Booster,Active,-73.7,77.0,2025-12-23
1,SITE0002,DEL-NW-0002,Delhi,Delhi NCR,Macro Tower,Active,-86.3,48.8,2025-06-09
2,SITE0003,LUC-NW-0003,Lucknow,UP East,Macro Tower,Under Maintenance,-107.8,42.2,2025-11-11



network (1300, 10)


,network_report_id,cell_site_id,city,circle,report_date,issue_type,severity,affected_customers,avg_downtime_minutes,current_status
0,NW000001,SITE0964,Indore,Madhya Pradesh,2026-05-25,Indoor coverage issue,critical,891,517,Monitoring
1,NW000002,SITE0972,Chennai,Tamil Nadu,2026-05-15,Tower outage,Critical,1590,447,Resolved
2,NW000003,SITE0646,Surat,Gujarat,2026-02-01,Fiber cut,Low,170,252,Monitoring



recharge (1500, 9)


,recharge_id,complaint_id,customer_id,recharge_date,recharge_amount,payment_mode,recharge_status,refund_status,refund_completed_date
0,RCH000001,CMP002599,CUST00947,2026-02-07,None,upi,failed payment,pending refund,None
1,RCH000002,CMP001502,CUST00264,2026-05-29,None,upi,Recharge Pending,Not Initiated,None
2,RCH000003,CMP004726,CUST00498,2026-03-30,None,Cash,FAILED,Failed,None



requests (1500, 10)


,service_request_id,complaint_id,customer_id,request_created_date,request_closed_date,request_type,service_status,assigned_team,resolution_time_hours,escalation_flag
0,SR000001,CMP004694,CUST00217,2026-04-06 02:00,,Recharge Support,Escalated,Recharge Ops,,Y
1,SR000002,CMP001907,CUST00813,2026-03-15 00:00,,Recharge Support,Open,Fiber Field Team,56,yes
2,SR000003,CMP003243,CUST00377,2026-05-07 21:00,2026-05-08 08:00,Billing Investigation,Closed,Care Escalation Desk,,N



Operations notes (first 300 characters):
001. Circle Ops observed repeated call drop complaints near Ahmedabad commuter corridor.
002. Manager asks developer to validate orphan complaint and service request links before final reporting.
003. NOC recommends checking high capacity utilization sites in Kolkata.
004. Circle Ops observed repeat


## Milestone 03 · Validate and standardize complaint records

**From the team:** Quality assurance warns that missing issue labels, mixed-case statuses, and invalid timestamps make complaint-age comparisons unreliable.

**Your responsibility:** Deduplicate only by complaint_id; standardize categories and statuses; parse dates and monetary fields. Record missing customer/site references and unparseable dates without erasing valid multi-complaint histories.

**Handover evidence:** Create `complaints_clean`, `quality_summary` and normalized `issue_group` with complaint-level uniqueness.

In [5]:
# Milestone 03 — Validate and standardize complaint records
# Task 1: Deduplicate only by complaint_id; standardize categories and statuses; parse dates and monetary fields. Record missing customer/site references and unparseable dates without erasing valid multi-complaint histories.
# Task 2: Produce this required evidence: Create `complaints_clean`, `quality_summary` and normalized `issue_group` with complaint-level uniqueness.
# Implement your code below and display the requested outputs.
complaints_clean = complaints_df.copy()

status_text = (complaints_clean['complaint_status']
               .astype('string')
               .str.strip()                          # remove spaces at the ends
               .str.lower()                          # make everything lowercase
               .str.replace(r'[-_\s]+', ' ', regex=True))   # '-' or extra spaces -> one space

status_map = {
    'open': 'Open',
    'in progress': 'In Progress',
    'pending': 'Pending',
    'pending review': 'Pending Review',
    'escalated': 'Escalated',
    'resolved': 'Resolved',
    'closed': 'Closed',
}

complaints_clean['complaint_status'] = status_text.map(status_map)   # anything else becomes NaN

display(complaints_clean['complaint_status'].value_counts(dropna=False))
category_text = (complaints_clean['issue_category']
                 .astype('string')
                 .str.strip()
                 .str.lower()
                 .str.replace(r'[-_\s]+', ' ', regex=True))

group_map = {
    'network down': 'Network Outage',
    'network outage': 'Network Outage',
    'fiber outage': 'Network Outage',
    'tower issue': 'Network Outage',
    'no signal': 'Signal / Call Quality',
    'weak signal': 'Signal / Call Quality',
    'call drop': 'Signal / Call Quality',
    'slow data': 'Data / Internet',
    'data speed low': 'Data / Internet',
    'internet not working': 'Data / Internet',
    'recharge failed': 'Recharge / Payment',
    'payment deducted recharge not done': 'Recharge / Payment',
    'refund not received': 'Recharge / Payment',
    'billing issue': 'Recharge / Payment',
    'service request pending': 'Service / Account',
    'sim activation pending': 'Service / Account',
    'complaint unresolved': 'Service / Account',
}

complaints_clean['issue_group'] = category_text.map(group_map).fillna('Unknown')

display(complaints_clean['issue_group'].value_counts(dropna=False))
dupes = complaints_clean[complaints_clean['complaint_id'].duplicated(keep=False)]
print("Rows involved in duplication:", len(dupes))
display(dupes.sort_values('complaint_id').head(10))
dup_mask = complaints_clean['complaint_id'].duplicated(keep='first')

duplicate_rows_removed = complaints_clean[dup_mask].copy()   # keep a record of what we drop
complaints_clean = complaints_clean[~dup_mask].copy()        # keep the first row of each ID

print("Rows removed:", len(duplicate_rows_removed))
print("Rows remaining:", len(complaints_clean))
print("complaint_id unique?", complaints_clean['complaint_id'].is_unique)
raw_dates = complaints_clean['complaint_date'].copy()

complaints_clean['complaint_date'] = pd.to_datetime(raw_dates, errors='coerce')

# rows that HAD a value but it could not be parsed (not the same as simply missing)
invalid_date_mask = raw_dates.notna() & complaints_clean['complaint_date'].isna()
print("Missing dates:", raw_dates.isna().sum())
print("Unparseable dates:", invalid_date_mask.sum())
display(raw_dates[invalid_date_mask].value_counts())

# amounts are already numbers, but make sure and check
complaints_clean['amount_impacted'] = pd.to_numeric(complaints_clean['amount_impacted'], errors='coerce')
print(complaints_clean['amount_impacted'].describe())
print(complaints_clean.dtypes[['complaint_date', 'amount_impacted']])
unknown_customer = ~complaints_clean['customer_id'].isin(customers_df['customer_id'])
unknown_site = ~complaints_clean['cell_site_id'].isin(cell_sites_df['cell_site_id'])

# flag columns, so these rows are marked for review rather than deleted
complaints_clean['missing_customer_ref'] = unknown_customer
complaints_clean['missing_site_ref'] = unknown_site
complaints_clean['invalid_date_flag'] = invalid_date_mask.reindex(complaints_clean.index, fill_value=False)

quality_summary = pd.DataFrame({
    'check': [
        'Duplicate complaint_id rows removed',
        'Missing complaint_status',
        'Missing issue_category (issue_group = Unknown)',
        'Missing amount_impacted',
        'Missing complaint_date',
        'Unparseable complaint_date',
        'Customer ID not found in customers reference',
        'Cell site ID not found in cell_sites reference',
    ],
    'row_count': [
        len(duplicate_rows_removed),
        complaints_clean['complaint_status'].isna().sum(),
        (complaints_clean['issue_group'] == 'Unknown').sum(),
        complaints_clean['amount_impacted'].isna().sum(),
        complaints_clean['complaint_date'].isna().sum() - complaints_clean['invalid_date_flag'].sum(),
        complaints_clean['invalid_date_flag'].sum(),
        unknown_customer.sum(),
        unknown_site.sum(),
    ],
})
display(quality_summary)
print("Rows in complaints_clean:", len(complaints_clean))


,count
complaint_status,
Closed,942
Open,880
In Progress,865
Pending Review,477
Escalated,462
NaN,461
Pending,457
Resolved,456


,count
issue_group,
Recharge / Payment,1211
Network Outage,1133
Signal / Call Quality,882
Data / Internet,873
Service / Account,611
Unknown,290


Rows involved in duplication: 100


,complaint_id,customer_id,cell_site_id,complaint_date,complaint_time,city,circle,complaint_channel,issue_category,issue_description,complaint_status,sentiment_tag,network_type,device_type,amount_impacted,issue_group
23,CMP001502,CUST00264,SITE0826,2026-02-24,11:04,Kolkata,West Bengal,Call Center,No Signal,Repeated complaint from same location and serv...,Open,Frustrated,4G,Android,0.0,Signal / Call Quality
3456,CMP001502,CUST00176,SITE0027,2026-03-04,09:59,Indore,Madhya Pradesh,MyJio App,fiber outage,Customer reports unstable network for the last...,In Progress,Angry,4G,JioPhone,399.0,Network Outage
3351,CMP001502,CUST00260,SITE0268,2026-05-05,11:00,Mumbai,Mumbai,WhatsApp Bot,payment deducted recharge not done,Payment was debited but recharge benefit is no...,Closed,Polite,4G,Fiber Router,0.0,Recharge / Payment
3340,CMP001502,CUST00556,SITE0194,2026-05-30,15:39,Indore,Madhya Pradesh,Email,NaN,Payment was debited but recharge benefit is no...,Escalated,Repeated Complaint,VoLTE,Fiber Router,399.0,Unknown
3321,CMP001502,CUST00142,SITE0751,2026-05-28,16:49,Jaipur,Rajasthan,MyJio App,No Signal,Customer is unable to use data indoors.,Resolved,Urgent,Fiber,iOS,199.0,Signal / Call Quality
3181,CMP001502,CUST00288,SITE0218,2026-05-16,23:16,Lucknow,UP East,Retail Store,data speed low,Fiber connection outage reported for apartment...,Resolved,Neutral,4G,Fiber Router,299.0,Data / Internet
3161,CMP001502,CUST00647,SITE0548,2026-04-28,08:17,Hyderabad,Andhra Pradesh & Telangana,Call Center,recharge failed,Repeated complaint from same location and serv...,In Progress,Neutral,Fiber,JioPhone,799.0,Recharge / Payment
3078,CMP001502,CUST01049,SITE0547,2026-04-21,07:30,Kolkata,West Bengal,WhatsApp Bot,service request pending,SIM activation or verification is pending afte...,Closed,Neutral,4G,Hotspot,299.0,Service / Account
3049,CMP001502,CUST00964,SITE0201,2026-04-22,14:44,Surat,Gujarat,Call Center,payment deducted recharge not done,Repeated complaint from same location and serv...,Resolved,Angry,5G,Fiber Router,0.0,Recharge / Payment
3024,CMP001502,CUST00035,SITE0316,2026-05-10,10:33,Delhi,Delhi NCR,Email,NaN,Repeated complaint from same location and serv...,In Progress,Neutral,4G,Hotspot,399.0,Unknown


Rows removed: 99
Rows remaining: 4901
complaint_id unique? True
Missing dates: 6
Unparseable dates: 6


,count
complaint_date,
2026-15-01,3
not_a_date,3


count    4476.000000
mean      271.051609
std       253.909086
min         0.000000
25%         0.000000
50%       239.000000
75%       399.000000
max       799.000000
Name: amount_impacted, dtype: float64
complaint_date     datetime64[ns]
amount_impacted           float64
dtype: object


,check,row_count
0,Duplicate complaint_id rows removed,99
1,Missing complaint_status,452
2,Missing issue_category (issue_group = Unknown),282
3,Missing amount_impacted,425
4,Missing complaint_date,6
5,Unparseable complaint_date,6
6,Customer ID not found in customers reference,35
7,Cell site ID not found in cell_sites reference,29


Rows in complaints_clean: 4901


## Milestone 04 · Prepare reliable network incident signals

**From the team:** Network operations wants tower status and incident severity contextualized before escalation decisions.

**Your responsibility:** Deduplicate network report IDs; normalize severity/status; aggregate distinct incidents at cell-site level; join site inventory without multiplying incidents.

**Handover evidence:** Create `network_clean`, `network_by_site`, and `sites_clean` with unique site IDs.

In [6]:
# Milestone 04 — Prepare reliable network incident signals
# Task 1: Deduplicate network report IDs; normalize severity/status; aggregate distinct incidents at cell-site level; join site inventory without multiplying incidents.
# Task 2: Produce this required evidence: Create `network_clean`, `network_by_site`, and `sites_clean` with unique site IDs.
# Implement your code below and display the requested outputs.
# Milestone 04 — Prepare reliable network incident signals

def tidy(series):
    """Lowercase, trim and collapse spaces/dashes so different spellings match."""
    return (series.astype('string')
                  .str.strip()
                  .str.lower()
                  .str.replace(r'[-_\s]+', ' ', regex=True))

# ---------- 1. Clean the network reports ----------
network_clean = network_df.copy()
network_clean['cell_site_id'] = network_clean['cell_site_id'].astype('string').str.strip().str.upper()
network_clean = network_clean.drop_duplicates(subset='network_report_id', keep='first').copy()

severity_map = {'critical': 'Critical', 'major outage': 'Major Outage',
                'high': 'High', 'medium': 'Medium', 'low': 'Low'}
net_status_map = {'open': 'Open', 'monitoring': 'Monitoring', 'escalated': 'Escalated',
                  'pending': 'Pending', 'resolved': 'Resolved', 'closed': 'Closed'}

network_clean['severity'] = tidy(network_clean['severity']).map(severity_map)
network_clean['current_status'] = tidy(network_clean['current_status']).map(net_status_map)
network_clean['issue_type'] = tidy(network_clean['issue_type']).str.title()
network_clean['report_date'] = pd.to_datetime(network_clean['report_date'], errors='coerce')
for col in ['affected_customers', 'avg_downtime_minutes']:
    network_clean[col] = pd.to_numeric(network_clean[col], errors='coerce')

# True/False flags: open = not Resolved/Closed, severe = Critical or Major Outage
network_clean['is_open'] = ~network_clean['current_status'].isin(['Resolved', 'Closed'])
network_clean['is_severe'] = network_clean['severity'].isin(['Critical', 'Major Outage'])
network_clean['is_open_severe'] = network_clean['is_open'] & network_clean['is_severe']

# ---------- 2. Clean the site inventory ----------
sites_clean = cell_sites_df.copy()
sites_clean['cell_site_id'] = sites_clean['cell_site_id'].astype('string').str.strip().str.upper()
sites_clean = sites_clean.drop_duplicates(subset='cell_site_id', keep='first').copy()
for col in ['site_status', 'tower_type', 'city', 'circle']:
    sites_clean[col] = tidy(sites_clean[col]).str.title()
sites_clean['last_maintenance_date'] = pd.to_datetime(sites_clean['last_maintenance_date'], errors='coerce')
for col in ['avg_signal_strength_dbm', 'capacity_utilization_pct']:
    sites_clean[col] = pd.to_numeric(sites_clean[col], errors='coerce')

# ---------- 3. Summarise incidents per cell site (one row per site) ----------
reports_with_site = network_clean.dropna(subset=['cell_site_id'])

network_by_site = (reports_with_site.groupby('cell_site_id')
    .agg(network_report_count=('network_report_id', 'size'),
         open_report_count=('is_open', 'sum'),
         severe_network_report_count=('is_open_severe', 'sum'),   # OPEN severe reports
         total_affected_customers=('affected_customers', 'sum'),
         avg_downtime_minutes=('avg_downtime_minutes', 'mean'),
         latest_report_date=('report_date', 'max'))
    .reset_index())

# ---------- 4. Join site details WITHOUT multiplying rows ----------
network_by_site = network_by_site.merge(
    sites_clean[['cell_site_id', 'site_name', 'city', 'circle', 'tower_type', 'site_status',
                 'avg_signal_strength_dbm', 'capacity_utilization_pct', 'last_maintenance_date']],
    on='cell_site_id', how='left', validate='one_to_one')   # errors if a site repeats

# ---------- 5. Reports pointing to sites we don't have (kept for review) ----------
orphan_network_reports = network_clean[~network_clean['cell_site_id'].isin(sites_clean['cell_site_id'])]

# ---------- 6. Evidence ----------
print("Network reports:", len(network_clean), "| unique IDs:", network_clean['network_report_id'].is_unique)
print("Bad/missing report dates:", network_clean['report_date'].isna().sum())
display(network_clean['severity'].value_counts(dropna=False))
display(network_clean['current_status'].value_counts(dropna=False))
print("Open:", network_clean['is_open'].sum(),
      "| Severe:", network_clean['is_severe'].sum(),
      "| Open and severe:", network_clean['is_open_severe'].sum())

print("\nSites in sites_clean:", len(sites_clean), "| unique site IDs:", sites_clean['cell_site_id'].is_unique)
display(sites_clean['site_status'].value_counts(dropna=False))

print("\nSites with incidents (network_by_site rows):", len(network_by_site))
print("Reports counted in network_by_site:", network_by_site['network_report_count'].sum())
print("Reports pointing to unknown sites:", len(orphan_network_reports))
display(network_by_site.sort_values('network_report_count', ascending=False).head(5))


Network reports: 1300 | unique IDs: True
Bad/missing report dates: 8


,count
severity,
High,381
Critical,371
Low,201
Medium,174
Major Outage,173


,count
current_status,
Open,247
Monitoring,234
Escalated,222
Resolved,210
Pending,201
Closed,186


Open: 904 | Severe: 544 | Open and severe: 391

Sites in sites_clean: 1000 | unique site IDs: True


,count
site_status,
Active,496
Planned Upgrade,187
Under Maintenance,173
Congested,144



Sites with incidents (network_by_site rows): 721
Reports counted in network_by_site: 1300
Reports pointing to unknown sites: 30


,cell_site_id,network_report_count,open_report_count,severe_network_report_count,total_affected_customers,avg_downtime_minutes,latest_report_date,site_name,city,circle,tower_type,site_status,avg_signal_strength_dbm,capacity_utilization_pct,last_maintenance_date
547,SITE0792,6,5,1,8852,394.833333,2026-05-28,CHE-NW-0792,Chennai,Tamil Nadu,Macro Tower,Congested,-90.4,90.5,2026-03-31
93,SITE0136,6,6,2,3800,548.000000,2026-06-01,PUN-NW-0136,Pune,Maharashtra,Indoor Booster,Congested,-81.5,70.3,2026-04-25
262,SITE0375,6,5,1,5871,458.500000,2026-04-25,MUM-NW-0375,Mumbai,Mumbai,Rooftop,Under Maintenance,-110.2,97.0,2026-05-09
285,SITE0404,6,5,2,4896,275.333333,2026-05-14,HYD-NW-0404,Hyderabad,Andhra Pradesh & Telangana,Small Cell,Active,-110.9,92.6,2025-08-06
357,SITE0517,5,5,3,4345,421.800000,2026-05-20,IND-NW-0517,Indore,Madhya Pradesh,Indoor Booster,Under Maintenance,-113.7,56.6,2025-08-23


## Milestone 05 · Audit recharge failures and refund risk

**From the team:** Recharge Support needs to spot unresolved money-impacting cases and failed payments linked to complaints.

**Your responsibility:** Normalize recharge status, parse amounts and dates, flag failed/pending outcomes and pending refunds, and aggregate by complaint ID. Do not discard valid multiple recharge events.

**Handover evidence:** Create `recharge_clean`, `recharge_by_complaint` and readable risk counts.

In [7]:
# Milestone 05 — Audit recharge failures and refund risk
# Task 1: Normalize recharge status, parse amounts and dates, flag failed/pending outcomes and pending refunds, and aggregate by complaint ID. Do not discard valid multiple recharge events.
# Task 2: Produce this required evidence: Create `recharge_clean`, `recharge_by_complaint` and readable risk counts.
# Implement your code below and display the requested outputs.
# Milestone 05 — Audit recharge failures and refund risk

# ---------- 1. Clean IDs and text columns ----------
recharge_clean = recharge_df.copy()
recharge_clean['recharge_id'] = recharge_clean['recharge_id'].astype('string').str.strip().str.upper()
recharge_clean = recharge_clean.drop_duplicates(subset='recharge_id', keep='first').copy()
for col in ['complaint_id', 'customer_id']:
    recharge_clean[col] = recharge_clean[col].astype('string').str.strip().str.upper()

recharge_status_map = {'failed': 'Failed', 'failed payment': 'Failed',
                       'payment deducted': 'Payment Deducted',
                       'recharge pending': 'Pending', 'success reversed': 'Success Reversed'}
refund_status_map = {'pending': 'Pending', 'pending refund': 'Pending',
                     'not initiated': 'Not Initiated', 'failed': 'Refund Failed',
                     'completed': 'Completed'}

recharge_clean['recharge_status'] = tidy(recharge_clean['recharge_status']).map(recharge_status_map)
recharge_clean['refund_status'] = tidy(recharge_clean['refund_status']).map(refund_status_map)
recharge_clean['payment_mode'] = tidy(recharge_clean['payment_mode']).str.title()

# ---------- 2. Amounts: remove symbols like ₹, convert to numbers, fix negatives ----------
raw_amount = (recharge_clean['recharge_amount'].astype('string')
              .str.replace(r'[^0-9.\-]', '', regex=True))
recharge_clean['recharge_amount'] = pd.to_numeric(raw_amount, errors='coerce')
recharge_clean['negative_amount_flag'] = recharge_clean['recharge_amount'] < 0   # keep a record
recharge_clean['recharge_amount'] = recharge_clean['recharge_amount'].abs()

# ---------- 3. Dates ----------
for col in ['recharge_date', 'refund_completed_date']:
    recharge_clean[col] = pd.to_datetime(recharge_clean[col], errors='coerce')

# ---------- 4. Risk flags ----------
recharge_clean['failed_or_pending_flag'] = recharge_clean['recharge_status'].isin(['Failed', 'Pending'])
recharge_clean['unresolved_refund_flag'] = recharge_clean['refund_status'].isin(
    ['Pending', 'Not Initiated', 'Refund Failed'])
recharge_clean['recharge_risk_flag'] = recharge_clean['unresolved_refund_flag']

# ---------- 5. One row per complaint (keeps every recharge event in the counts) ----------
recharge_by_complaint = (
    recharge_clean.dropna(subset=['complaint_id'])
    .assign(unresolved_refund_amount=lambda d: d['recharge_amount'].where(d['unresolved_refund_flag'], 0))
    .groupby('complaint_id')
    .agg(recharge_count=('recharge_id', 'size'),
         failed_recharge_count=('failed_or_pending_flag', 'sum'),
         recharge_risk_count=('recharge_risk_flag', 'sum'),
         total_recharge_amount=('recharge_amount', 'sum'),
         unresolved_refund_amount=('unresolved_refund_amount', 'sum'),
         latest_recharge_date=('recharge_date', 'max'))
    .reset_index())

# ---------- 6. Recharges with no matching complaint (kept for review, not discarded) ----------
orphan_recharge = recharge_clean[~recharge_clean['complaint_id'].isin(complaints_clean['complaint_id'])]

# ---------- 7. Evidence ----------
print("Recharge records:", len(recharge_clean), "| unique recharge IDs:", recharge_clean['recharge_id'].is_unique)
print("Missing amounts:", recharge_clean['recharge_amount'].isna().sum(),
      "| negative amounts fixed:", recharge_clean['negative_amount_flag'].sum(),
      "| bad/missing recharge dates:", recharge_clean['recharge_date'].isna().sum())

print("\nRecharge status counts:")
display(recharge_clean['recharge_status'].value_counts(dropna=False))
print("Refund status counts:")
display(recharge_clean['refund_status'].value_counts(dropna=False))

print("Failed or pending recharges:", recharge_clean['failed_or_pending_flag'].sum(),
      "| unresolved refunds:", recharge_clean['unresolved_refund_flag'].sum())

matched = recharge_by_complaint[recharge_by_complaint['complaint_id'].isin(complaints_clean['complaint_id'])]
print("\nComplaints with recharge history:", len(matched),
      "| with at least one unresolved refund:", (matched['recharge_risk_count'] > 0).sum())
print("Amount tied to unresolved refunds (matched complaints): Rs.", matched['unresolved_refund_amount'].sum())
print("Recharges counted in recharge_by_complaint:", recharge_by_complaint['recharge_count'].sum())
print("Recharge rows with no matching complaint:", len(orphan_recharge))

display(recharge_by_complaint.sort_values('unresolved_refund_amount', ascending=False).head(5))


Recharge records: 1500 | unique recharge IDs: True
Missing amounts: 252 | negative amounts fixed: 128 | bad/missing recharge dates: 10

Recharge status counts:


,count
recharge_status,
Failed,641
Payment Deducted,228
Pending,217
Success Reversed,209
NaN,205


Refund status counts:


,count
refund_status,
Pending,514
Not Initiated,279
Refund Failed,244
Completed,244
NaN,219


Failed or pending recharges: 858 | unresolved refunds: 1037

Complaints with recharge history: 1253 | with at least one unresolved refund: 924
Amount tied to unresolved refunds (matched complaints): Rs. 341041
Recharges counted in recharge_by_complaint: 1500
Recharge rows with no matching complaint: 4


,complaint_id,recharge_count,failed_recharge_count,recharge_risk_count,total_recharge_amount,unresolved_refund_amount,latest_recharge_date
371,CMP001502,39,22,23,9088,4812,2026-05-29
338,CMP001377,2,0,2,1598,1598,2026-05-04
1015,CMP004098,3,1,2,1797,1598,2026-05-30
1223,CMP004882,2,1,2,1598,1598,2026-03-07
806,CMP003215,2,1,2,1398,1398,2026-04-16


## Milestone 06 · Detect service request and SLA follow-ups

**From the team:** The Service Desk wants unresolved or escalated requests visible on the original complaint without introducing duplicate complaint rows.

**Your responsibility:** Normalize service request statuses, parse request dates and resolution hours, distinguish still-open from closed cases, and aggregate by complaint before merging.

**Handover evidence:** Create `requests_clean`, `requests_by_complaint`; show unresolved and escalated counts and orphan references.

In [8]:
# Milestone 06 — Detect service request and SLA follow-ups
# Task 1: Normalize service request statuses, parse request dates and resolution hours, distinguish still-open from closed cases, and aggregate by complaint before merging.
# Task 2: Produce this required evidence: Create `requests_clean`, `requests_by_complaint`; show unresolved and escalated counts and orphan references.
# Implement your code below and display the requested outputs.
# Milestone 06 — Detect service request and SLA follow-ups

SLA_THRESHOLD_HOURS = 72     # assumption: no SLA is documented in the data, so this is our own rule

# ---------- 1. Clean IDs and text columns ----------
requests_clean = requests_df.copy()
requests_clean['service_request_id'] = requests_clean['service_request_id'].astype('string').str.strip().str.upper()
requests_clean = requests_clean.drop_duplicates(subset='service_request_id', keep='first').copy()
for col in ['complaint_id', 'customer_id']:
    requests_clean[col] = requests_clean[col].astype('string').str.strip().str.upper()

service_status_map = {'open': 'Open', 'pending': 'Pending', 'in progress': 'In Progress',
                      'escalated': 'Escalated', 'resolved': 'Resolved', 'closed': 'Closed'}
team_map = {'fiber field team': 'Fiber Field Team', 'network noc': 'Network NOC',
            'recharge ops': 'Recharge Ops', 'circle operations': 'Circle Operations',
            'care escalation desk': 'Care Escalation Desk', 'retail ops': 'Retail Ops'}

requests_clean['service_status'] = tidy(requests_clean['service_status']).map(service_status_map)  # blanks -> NaN
requests_clean['assigned_team'] = tidy(requests_clean['assigned_team']).map(team_map)
requests_clean['request_type'] = tidy(requests_clean['request_type']).str.title()

# Yes / Y -> True, No / N -> False, blank stays missing
requests_clean['escalation_flag'] = tidy(requests_clean['escalation_flag']).map(
    {'yes': True, 'y': True, 'no': False, 'n': False})

# ---------- 2. Dates and resolution hours ----------
for col in ['request_created_date', 'request_closed_date']:
    requests_clean[col] = pd.to_datetime(requests_clean[col].replace('', pd.NA), errors='coerce')

reported_hours = pd.to_numeric(requests_clean['resolution_time_hours'].replace('', pd.NA), errors='coerce')
calculated_hours = (requests_clean['request_closed_date'] - requests_clean['request_created_date']).dt.total_seconds() / 3600
requests_clean['resolution_hours'] = reported_hours.fillna(calculated_hours)   # use recorded hours, else closed - created

# ---------- 3. Flags ----------
requests_clean['is_unresolved'] = requests_clean['service_status'].isin(['Open', 'Pending', 'In Progress', 'Escalated'])
requests_clean['is_escalated'] = (requests_clean['escalation_flag'].astype('boolean').fillna(False).astype(bool)
                                  | (requests_clean['service_status'] == 'Escalated'))
requests_clean['sla_review_flag'] = requests_clean['resolution_hours'] > SLA_THRESHOLD_HOURS

# ---------- 4. One row per complaint ----------
requests_by_complaint = (requests_clean.dropna(subset=['complaint_id'])
    .groupby('complaint_id')
    .agg(request_count=('service_request_id', 'size'),
         unresolved_request_count=('is_unresolved', 'sum'),
         open_request_count=('is_unresolved', 'sum'),
         escalated_request_count=('is_escalated', 'sum'),
         sla_review_count=('sla_review_flag', 'sum'),
         max_resolution_hours=('resolution_hours', 'max'),
         latest_request_date=('request_created_date', 'max'))
    .reset_index())

# ---------- 5. Requests with no matching complaint (kept for review) ----------
orphan_requests = requests_clean[~requests_clean['complaint_id'].isin(complaints_clean['complaint_id'])]

# ---------- 6. Evidence ----------
print("Service requests:", len(requests_clean), "| unique IDs:", requests_clean['service_request_id'].is_unique)
print("Missing service_status:", requests_clean['service_status'].isna().sum(),
      "| resolution hours known:", requests_clean['resolution_hours'].notna().sum())

print("\nService status counts:")
display(requests_clean['service_status'].value_counts(dropna=False))

print("Unresolved requests:", requests_clean['is_unresolved'].sum(),
      "| escalated:", requests_clean['is_escalated'].sum(),
      f"| over {SLA_THRESHOLD_HOURS}h SLA:", requests_clean['sla_review_flag'].sum())

print("\nWorkload by assigned team:")
team_workload = (requests_clean.groupby('assigned_team')
    .agg(total_requests=('service_request_id', 'size'),
         unresolved=('is_unresolved', 'sum'),
         escalated=('is_escalated', 'sum'),
         sla_review=('sla_review_flag', 'sum'),
         avg_resolution_hours=('resolution_hours', 'mean'))
    .round(1).sort_values('unresolved', ascending=False))
display(team_workload)

matched = requests_by_complaint[requests_by_complaint['complaint_id'].isin(complaints_clean['complaint_id'])]
print("Complaints with service requests:", len(matched),
      "| with unresolved:", (matched['unresolved_request_count'] > 0).sum(),
      "| with escalated:", (matched['escalated_request_count'] > 0).sum(),
      "| with SLA review:", (matched['sla_review_count'] > 0).sum())
print("Requests counted in requests_by_complaint:", requests_by_complaint['request_count'].sum())
print("Requests with no matching complaint:", len(orphan_requests))
display(orphan_requests[['service_request_id', 'complaint_id', 'customer_id', 'service_status']])


Service requests: 1500 | unique IDs: True
Missing service_status: 232 | resolution hours known: 776

Service status counts:


,count
service_status,
NaN,232
Open,227
Escalated,218
Closed,214
Pending,207
In Progress,205
Resolved,197


Unresolved requests: 857 | escalated: 840 | over 72h SLA: 280

Workload by assigned team:


,total_requests,unresolved,escalated,sla_review,avg_resolution_hours
assigned_team,,,,,
Fiber Field Team,280,171,165,46,53.9
Network NOC,258,155,158,52,60.4
Recharge Ops,249,139,140,37,54.6
Circle Operations,249,139,138,56,60.5
Retail Ops,227,127,116,39,53.4
Care Escalation Desk,237,126,123,50,60.1


Complaints with service requests: 1262 | with unresolved: 776 | with escalated: 762 | with SLA review: 271
Requests counted in requests_by_complaint: 1500
Requests with no matching complaint: 3


,service_request_id,complaint_id,customer_id,service_status
325,SR000326,CMPX943667,CUST00055,Pending
1138,SR001139,CMPX905803,CUST01012,NaN
1492,SR001493,CMPX928838,CUST00696,Open


## Milestone 07 · Analyze city, circle and repeated customer impact

**From the team:** Circle managers need geographic clusters and repeat contacts—not misleading counts created by multi-table joins.

**Your responsibility:** Use cleaned complaint facts and site context to summarize cities/circles and repeated customer complaints; preserve missing site references for review.

**Handover evidence:** Create `location_summary`, `repeat_customers` and `customer_repeat_count` at complaint grain.

In [9]:
# Milestone 07 — Analyze city, circle and repeated customer impact
# Task 1: Use cleaned complaint facts and site context to summarize cities/circles and repeated customer complaints; preserve missing site references for review.
# Task 2: Produce this required evidence: Create `location_summary`, `repeat_customers` and `customer_repeat_count` at complaint grain.
# Implement your code below and display the requested outputs.
# Milestone 07 — Analyze city, circle and repeated customer impact

# ---------- 1. Tidy ID and location text (strip stray spaces only) ----------
for col in ['customer_id', 'cell_site_id', 'city', 'circle']:
    complaints_clean[col] = complaints_clean[col].astype('string').str.strip()

# ---------- 2. Repeated customer complaints ----------
customer_repeat_count = (complaints_clean.groupby('customer_id')['complaint_id'].nunique()
                         .rename('customer_complaint_count').reset_index())     # one row per customer

repeat_customers = (customer_repeat_count[customer_repeat_count['customer_complaint_count'] >= 3]
                    .sort_values('customer_complaint_count', ascending=False)
                    .reset_index(drop=True))

# attach the count back to every complaint (many complaints -> one customer row, so no duplication)
complaint_level_repeat = complaints_clean[['complaint_id', 'customer_id']].merge(
    customer_repeat_count, on='customer_id', how='left', validate='many_to_one')

# ---------- 3. Simple complaint flags ----------
CONNECTIVITY_GROUPS = ['Network Outage', 'Signal / Call Quality', 'Data / Internet']
complaints_clean['is_unresolved_complaint'] = ~complaints_clean['complaint_status'].isin(['Resolved', 'Closed'])
complaints_clean['is_connectivity_complaint'] = complaints_clean['issue_group'].isin(CONNECTIVITY_GROUPS)

# ---------- 4. City / circle summary: complaints + network incidents ----------
complaint_location = (complaints_clean.groupby(['circle', 'city'])
    .agg(complaint_count=('complaint_id', 'size'),
         unique_customers=('customer_id', 'nunique'),
         unresolved_complaints=('is_unresolved_complaint', 'sum'),
         connectivity_complaints=('is_connectivity_complaint', 'sum'),
         unknown_site_complaints=('missing_site_ref', 'sum'))   # kept for review, not dropped
    .reset_index())

network_location = (network_clean
    .assign(city=network_clean['city'].astype('string').str.strip(),
            circle=network_clean['circle'].astype('string').str.strip())
    .groupby(['circle', 'city'])
    .agg(network_reports=('network_report_id', 'size'),
         open_severe_reports=('is_open_severe', 'sum'),
         affected_customers=('affected_customers', 'sum'))
    .reset_index())

location_summary = (complaint_location
    .merge(network_location, on=['circle', 'city'], how='left', validate='one_to_one')
    .sort_values('complaint_count', ascending=False).reset_index(drop=True))

# ---------- 5. Cell-site hot spots ----------
site_complaints = (complaints_clean.groupby('cell_site_id')
    .agg(complaint_count=('complaint_id', 'size'), missing_site_ref=('missing_site_ref', 'max'))
    .reset_index()
    .merge(network_by_site[['cell_site_id', 'city', 'network_report_count', 'severe_network_report_count']],
           on='cell_site_id', how='left', validate='one_to_one')
    .sort_values('complaint_count', ascending=False))

# ---------- 6. Evidence ----------
print("Customers:", len(customer_repeat_count),
      "| repeat customers (3+ complaints):", len(repeat_customers))
print("Complaints belonging to repeat customers:",
      (complaint_level_repeat['customer_complaint_count'] >= 3).sum(), "of", len(complaint_level_repeat))
print("Unresolved complaints:", complaints_clean['is_unresolved_complaint'].sum(),
      "| connectivity complaints:", complaints_clean['is_connectivity_complaint'].sum())

print("\nTop 10 customers by complaints:")
display(repeat_customers.head(10))

print("Location summary (city within circle):")
display(location_summary)

print("Complaints by circle:")
display(location_summary.groupby('circle')[['complaint_count', 'unresolved_complaints', 'open_severe_reports']]
        .sum().sort_values('complaint_count', ascending=False))

print("Top 10 cell sites by complaints (missing_site_ref = site not in site list):")
display(site_complaints.head(10))
print("Complaints counted in location_summary:", location_summary['complaint_count'].sum(),
      "| network reports counted:", location_summary['network_reports'].sum())


Customers: 1217 | repeat customers (3+ complaints): 926
Complaints belonging to repeat customers: 4438 of 4901
Unresolved complaints: 3531 | connectivity complaints: 2841

Top 10 customers by complaints:


,customer_id,customer_complaint_count
0,CUST00597,11
1,CUST00615,11
2,CUST01170,11
3,CUST00097,10
4,CUST00005,10
5,CUST00169,10
6,CUST00527,10
7,CUST00504,10
8,CUST01131,10
9,CUST01135,10


Location summary (city within circle):


,circle,city,complaint_count,unique_customers,unresolved_complaints,connectivity_complaints,unknown_site_complaints,network_reports,open_severe_reports,affected_customers
0,Madhya Pradesh,Indore,452,387,321,256,4,130,40,124095
1,Maharashtra,Pune,447,379,315,250,2,116,32,100616
2,Karnataka,Bengaluru,441,364,313,265,5,141,46,129229
3,Delhi NCR,Delhi,440,365,318,272,4,105,30,86133
4,Tamil Nadu,Chennai,430,356,314,246,1,114,34,108163
5,Gujarat,Surat,428,368,308,240,1,109,33,91959
6,Mumbai,Mumbai,418,345,299,239,1,99,28,101189
7,West Bengal,Kolkata,418,360,308,257,3,96,38,81839
8,Andhra Pradesh & Telangana,Hyderabad,381,326,273,216,1,115,36,95484
9,Rajasthan,Jaipur,368,321,276,212,1,99,27,93068


Complaints by circle:


,complaint_count,unresolved_complaints,open_severe_reports
circle,,,
Gujarat,779,564,52
Madhya Pradesh,452,321,40
Maharashtra,447,315,32
Karnataka,441,313,46
Delhi NCR,440,318,30
Tamil Nadu,430,314,34
Mumbai,418,299,28
West Bengal,418,308,38
Andhra Pradesh & Telangana,381,273,36


Top 10 cell sites by complaints (missing_site_ref = site not in site list):


,cell_site_id,complaint_count,missing_site_ref,city,network_report_count,severe_network_report_count
26,SITE0027,14,False,Indore,2.0,0.0
681,SITE0686,13,False,<NA>,NaN,NaN
943,SITE0951,12,False,Kolkata,3.0,1.0
754,SITE0760,12,False,Indore,3.0,0.0
808,SITE0816,12,False,<NA>,NaN,NaN
630,SITE0635,12,False,Kolkata,2.0,1.0
569,SITE0574,12,False,Ahmedabad,1.0,0.0
713,SITE0719,12,False,Indore,2.0,1.0
32,SITE0033,12,False,<NA>,NaN,NaN
980,SITE0989,11,False,Ahmedabad,2.0,0.0


Complaints counted in location_summary: 4901 | network reports counted: 1300


## Milestone 08 · Build the complaint-level investigation view

**From the team:** CareOps needs one dependable record per original complaint with network, customer, recharge and service evidence together.

**Your responsibility:** Merge customer and site dimensions plus complaint-aggregated evidence, preserve complaints without matches, fill only count metrics with zero, and reconcile row counts and IDs.

**Handover evidence:** Create `final_cases_df` with exactly one record for every distinct complaint.

In [10]:
# Milestone 08 — Build the complaint-level investigation view
# Task 1: Merge customer and site dimensions plus complaint-aggregated evidence, preserve complaints without matches, fill only count metrics with zero, and reconcile row counts and IDs.
# Task 2: Produce this required evidence: Create `final_cases_df` with exactly one record for every distinct complaint.
# Implement your code below and display the requested outputs.
# Milestone 08 — Build the complaint-level investigation view

# ---------- 1. Clean the customer table ----------
customers_clean = customers_df.copy()
customers_clean['customer_id'] = customers_clean['customer_id'].astype('string').str.strip().str.upper()
customers_clean = customers_clean.drop_duplicates(subset='customer_id', keep='first').copy()
for col in ['customer_segment', 'plan_type', 'preferred_service']:
    customers_clean[col] = tidy(customers_clean[col]).str.title()
customers_clean['home_city'] = customers_clean['home_city'].astype('string').str.strip().str.title()
customers_clean['account_age_days'] = pd.to_numeric(customers_clean['account_age_days'], errors='coerce')
customers_clean['monthly_recharge_value'] = pd.to_numeric(customers_clean['monthly_recharge_value'], errors='coerce')
customers_clean = customers_clean.rename(columns={'circle': 'customer_circle'})   # avoid clash with complaint circle

# ---------- 2. Make sure every join key is formatted the same way ----------
for frame in (complaints_clean, customers_clean, sites_clean, network_by_site,
              recharge_by_complaint, requests_by_complaint, customer_repeat_count):
    for key in ['customer_id', 'cell_site_id', 'complaint_id']:
        if key in frame.columns:
            frame[key] = frame[key].astype('string').str.strip().str.upper()

# ---------- 3. Merge everything onto the complaints (left joins keep every complaint) ----------
base_rows = len(complaints_clean)
final_cases_df = complaints_clean.copy()

final_cases_df = final_cases_df.merge(customers_clean, on='customer_id',
                                      how='left', validate='many_to_one')

site_columns = ['cell_site_id', 'site_name', 'tower_type', 'site_status',
                'avg_signal_strength_dbm', 'capacity_utilization_pct', 'last_maintenance_date']
final_cases_df = final_cases_df.merge(sites_clean[site_columns], on='cell_site_id',
                                      how='left', validate='many_to_one')

network_columns = ['cell_site_id', 'network_report_count', 'open_report_count', 'severe_network_report_count',
                   'total_affected_customers', 'avg_downtime_minutes', 'latest_report_date']
final_cases_df = final_cases_df.merge(network_by_site[network_columns], on='cell_site_id',
                                      how='left', validate='many_to_one')

final_cases_df = final_cases_df.merge(recharge_by_complaint, on='complaint_id',
                                      how='left', validate='one_to_one')
final_cases_df = final_cases_df.merge(requests_by_complaint, on='complaint_id',
                                      how='left', validate='one_to_one')
final_cases_df = final_cases_df.merge(customer_repeat_count, on='customer_id',
                                      how='left', validate='many_to_one')

# ---------- 4. Fill ONLY the count columns with 0 (no records = zero events) ----------
count_columns = ['network_report_count', 'open_report_count', 'severe_network_report_count',
                 'total_affected_customers', 'recharge_count', 'failed_recharge_count',
                 'recharge_risk_count', 'request_count', 'unresolved_request_count',
                 'open_request_count',
                 'escalated_request_count', 'sla_review_count']
final_cases_df[count_columns] = final_cases_df[count_columns].fillna(0).astype(int)

# ---------- 5. Reconcile: prove nothing was lost or duplicated ----------
recharge_in_cases = recharge_clean[recharge_clean['complaint_id'].isin(final_cases_df['complaint_id'])]
requests_in_cases = requests_clean[requests_clean['complaint_id'].isin(final_cases_df['complaint_id'])]

assert len(final_cases_df) == base_rows == complaints_clean['complaint_id'].nunique(), "Row count changed!"
assert final_cases_df['complaint_id'].is_unique, "Duplicate complaint rows!"
assert final_cases_df['recharge_count'].sum() == len(recharge_in_cases), "Recharge events lost or duplicated!"
assert final_cases_df['request_count'].sum() == len(requests_in_cases), "Service requests lost or duplicated!"

print("Complaints before merging:", base_rows, "| rows after merging:", len(final_cases_df))
print("One row per complaint?", final_cases_df['complaint_id'].is_unique)
print("Columns:", final_cases_df.shape[1])

print("\nCoverage of each source:")
print("  Complaints with a customer record:   ", final_cases_df['customer_name'].notna().sum(),
      "| without:", final_cases_df['customer_name'].isna().sum())
print("  Complaints with a site record:       ", final_cases_df['site_name'].notna().sum(),
      "| without:", final_cases_df['site_name'].isna().sum())
print("  Complaints at sites with incidents:  ", (final_cases_df['network_report_count'] > 0).sum())
print("  Complaints with open severe incident:", (final_cases_df['severe_network_report_count'] > 0).sum())
print("  Complaints with recharge history:    ", (final_cases_df['recharge_count'] > 0).sum(),
      "| recharge events attached:", final_cases_df['recharge_count'].sum())
print("  Complaints with service requests:    ", (final_cases_df['request_count'] > 0).sum(),
      "| requests attached:", final_cases_df['request_count'].sum())

display(final_cases_df[['complaint_id', 'customer_id', 'city', 'issue_group', 'complaint_status',
                        'site_status', 'severe_network_report_count', 'recharge_risk_count',
                        'escalated_request_count', 'sla_review_count', 'customer_complaint_count']].head(8))


Complaints before merging: 4901 | rows after merging: 4901
One row per complaint? True
Columns: 55

Coverage of each source:
  Complaints with a customer record:    4866 | without: 35
  Complaints with a site record:        4872 | without: 29
  Complaints at sites with incidents:   3443
  Complaints with open severe incident: 1582
  Complaints with recharge history:     1253 | recharge events attached: 1496
  Complaints with service requests:     1262 | requests attached: 1497


,complaint_id,customer_id,city,issue_group,complaint_status,site_status,severe_network_report_count,recharge_risk_count,escalated_request_count,sla_review_count,customer_complaint_count
0,CMP000001,CUST00246,Jaipur,Unknown,Open,Planned Upgrade,1,0,0,0,4
1,CMP000002,CUST00793,Ahmedabad,Unknown,NaN,Under Maintenance,0,0,1,0,4
2,CMP000003,CUST00931,Hyderabad,Unknown,In Progress,Congested,0,1,0,1,2
3,CMP000004,CUST00043,Ahmedabad,Data / Internet,Open,Active,1,0,0,0,5
4,CMP000005,CUST00723,Chennai,Recharge / Payment,Closed,Active,0,0,0,0,3
5,CMP000006,CUST00668,Lucknow,Signal / Call Quality,Pending,Under Maintenance,1,0,0,0,6
6,CMP000007,CUST00745,Pune,Service / Account,Closed,Congested,1,1,0,0,6
7,CMP000008,CUST01112,Kolkata,Data / Internet,Pending Review,Under Maintenance,1,0,0,0,4


## Milestone 09 · Prioritize and recommend transparent next actions

**From the team:** Support leaders want explainable P0–P3 priorities based on four risk conditions, not an opaque model. They also need a draft message prompt without sending customer data externally.

**Your responsibility:** Assign P0 (3–4 signals), P1 (2), P2 (1), or P3 (0) using (1) open severe network incidents or down/inactive/outage site, (2) recharge risk, (3) escalated or SLA-review service requests, (4) three or more complaints by a customer. Produce case-specific next actions and offline AI prompt previews.

**Handover evidence:** Create four boolean risk signals, `case_priority`, `priority_score`, `recommended_action`, `ai_prompt_preview` in `final_cases_df`, and `priority_summary`.


In [11]:
# Milestone 09 — Prioritize and recommend transparent next actions
# Task 1: Create Boolean severe_network_signal: open severe network reports OR a site that is Down, Inactive, or Outage.
# Task 2: Create Boolean recharge_risk_signal from recharge_risk_count > 0.
# Task 3: Create Boolean service_followup_signal for escalated requests OR requests flagged for SLA review.
# Task 4: Create Boolean repeat_contact_signal for customers with at least three complaints.
# Task 5: Sum the four signals into priority_score and classify 3–4=P0, 2=P1, 1=P2, 0=P3.
# Task 6: Assign recommended_action by network, recharge, service and repeat-contact precedence.
# Task 7: Create privacy-safe ai_prompt_preview strings without calling an external API.
# Task 8: Create priority_summary with case_priority and case_count, and display priority breakdown and P0 cases.
# Milestone 09 — Prioritize and recommend transparent next actions

# ---------- 1. Four True/False risk signals ----------
DOWN_STATUSES = ['down', 'inactive', 'outage']
site_status_text = final_cases_df['site_status'].astype('string').str.strip().str.casefold()

final_cases_df['severe_network_signal'] = (
    (final_cases_df['severe_network_report_count'] > 0) | site_status_text.isin(DOWN_STATUSES).fillna(False)
).astype(bool)

final_cases_df['recharge_risk_signal'] = (final_cases_df['recharge_risk_count'] > 0).astype(bool)

final_cases_df['service_followup_signal'] = (
    (final_cases_df['escalated_request_count'] > 0) | (final_cases_df['sla_review_count'] > 0)
).astype(bool)

final_cases_df['repeat_contact_signal'] = (final_cases_df['customer_complaint_count'] >= 3).astype(bool)

# ---------- 2. Score and priority (P0 = 3-4 signals, P1 = 2, P2 = 1, P3 = 0) ----------
signal_columns = ['severe_network_signal', 'recharge_risk_signal',
                  'service_followup_signal', 'repeat_contact_signal']
final_cases_df['priority_score'] = final_cases_df[signal_columns].sum(axis=1).astype(int)

def score_to_priority(score):
    if score >= 3:
        return 'P0'
    if score == 2:
        return 'P1'
    if score == 1:
        return 'P2'
    return 'P3'

final_cases_df['case_priority'] = final_cases_df['priority_score'].map(score_to_priority)
priority_labels = {'P0': 'P0 - Critical', 'P1': 'P1 - High', 'P2': 'P2 - Medium', 'P3': 'P3 - Low'}
final_cases_df['priority_label'] = final_cases_df['case_priority'].map(priority_labels)

# ---------- 3. One primary action: Network > Recharge > Customer Care > repeat follow-up ----------
def choose_action(row):
    if row['severe_network_signal']:
        return 'Network Operations', 'Investigate the affected network site and update the customer'
    if row['recharge_risk_signal']:
        return 'Recharge Operations', 'Verify the failed recharge transaction and refund status'
    if row['service_followup_signal']:
        return 'Customer Care', 'Review the service request escalation and SLA status'
    if row['repeat_contact_signal']:
        return 'Customer Care - Repeat Follow-up', 'Contact the customer and investigate recurring service problems'
    return 'Standard Support', 'Continue the standard complaint-resolution process'

actions = final_cases_df.apply(choose_action, axis=1, result_type='expand')
final_cases_df['action_owner'] = actions[0]
final_cases_df['recommended_action'] = actions[1]

# ---------- 4. Case-specific explanation of why the case was flagged ----------
def describe_risks(row):
    parts = []
    if row['severe_network_signal']:
        parts.append(f"{row['severe_network_report_count']} open severe incident(s) at the site; "
                     f"site status {row['site_status']}")
    if row['recharge_risk_signal']:
        parts.append(f"{row['recharge_risk_count']} recharge(s) with unresolved refund")
    if row['service_followup_signal']:
        parts.append(f"{row['escalated_request_count']} escalated and "
                     f"{row['sla_review_count']} SLA-review request(s)")
    if row['repeat_contact_signal']:
        parts.append(f"customer has {row['customer_complaint_count']} complaints")
    return '; '.join(parts) if parts else 'No risk conditions met'

final_cases_df['risk_summary'] = final_cases_df.apply(describe_risks, axis=1)

# ---------- 5. Privacy-safe AI prompt text (no customer name or ID, no external API call) ----------
def build_prompt(row):
    status = row['complaint_status'] if pd.notna(row['complaint_status']) else 'Not recorded'
    return (
        "You are a Jio customer support assistant. Draft a short, polite update for a customer. "
        "Do not include personal details and do not promise refunds, fixes or resolution dates.\n"
        f"Case reference: {row['complaint_id']}\n"
        f"Location: {row['city']}, {row['circle']}\n"
        f"Issue type: {row['issue_group']}\n"
        f"Current status: {status}\n"
        f"Priority: {row['priority_label']}\n"
        f"Internal findings: {row['risk_summary']}\n"
        f"Next step for our team: {row['recommended_action']}"
    )

final_cases_df['ai_prompt_preview'] = final_cases_df.apply(build_prompt, axis=1)

# ---------- 6. Priority summary ----------
priority_summary = (final_cases_df.groupby('case_priority').size().rename('case_count')
                    .reset_index().sort_values('case_priority').reset_index(drop=True))
priority_summary['share_pct'] = (priority_summary['case_count'] / priority_summary['case_count'].sum() * 100).round(1)

# ---------- 7. Evidence ----------
print("Priority distribution:")
display(priority_summary)

print("Signals that are True (number of complaints):")
display(final_cases_df[signal_columns].sum().to_frame('complaints'))

print("Priority score counts (how many signals each complaint triggers):")
display(final_cases_df['priority_score'].value_counts().sort_index().to_frame('complaints'))

print("Action owner by priority:")
display(pd.crosstab(final_cases_df['case_priority'], final_cases_df['action_owner']))

print("P0 - Critical complaints needing immediate attention (top 15 by score):")
p0_cases = (final_cases_df[final_cases_df['case_priority'] == 'P0']
            .sort_values(['priority_score', 'customer_complaint_count'], ascending=False))
display(p0_cases[['complaint_id', 'city', 'issue_group', 'complaint_status', 'priority_score',
                  'action_owner', 'risk_summary']].head(15))

print("Sample AI prompt preview:\n")
print(final_cases_df['ai_prompt_preview'].iloc[0])


Priority distribution:


,case_priority,case_count,share_pct
0,P0,584,11.9
1,P1,1888,38.5
2,P2,2224,45.4
3,P3,205,4.2


Signals that are True (number of complaints):


,complaints
severe_network_signal,1582
recharge_risk_signal,924
service_followup_signal,868
repeat_contact_signal,4438


Priority score counts (how many signals each complaint triggers):


,complaints
priority_score,
0,205
1,2224
2,1888
3,524
4,60


Action owner by priority:


action_owner,Customer Care,Customer Care - Repeat Follow-up,Network Operations,Recharge Operations,Standard Support
case_priority,,,,,
P0,0,0,485,99,0
P1,427,0,987,474,0
P2,48,2025,110,41,0
P3,0,0,0,0,205


P0 - Critical complaints needing immediate attention (top 15 by score):


,complaint_id,city,issue_group,complaint_status,priority_score,action_owner,risk_summary
113,CMP000119,Bengaluru,Recharge / Payment,Resolved,4,Network Operations,1 open severe incident(s) at the site; site st...
1190,CMP001225,Pune,Signal / Call Quality,Closed,4,Network Operations,1 open severe incident(s) at the site; site st...
1435,CMP001472,Pune,Unknown,NaN,4,Network Operations,1 open severe incident(s) at the site; site st...
1328,CMP001364,Indore,Signal / Call Quality,Open,4,Network Operations,1 open severe incident(s) at the site; site st...
2471,CMP002525,Kolkata,Network Outage,Closed,4,Network Operations,1 open severe incident(s) at the site; site st...
2997,CMP003063,Hyderabad,Recharge / Payment,Pending,4,Network Operations,1 open severe incident(s) at the site; site st...
4432,CMP004521,Delhi,Network Outage,Pending,4,Network Operations,2 open severe incident(s) at the site; site st...
4467,CMP004556,Mumbai,Network Outage,Open,4,Network Operations,2 open severe incident(s) at the site; site st...
828,CMP000854,Mumbai,Recharge / Payment,In Progress,4,Network Operations,1 open severe incident(s) at the site; site st...
1864,CMP001907,Delhi,Recharge / Payment,Closed,4,Network Operations,1 open severe incident(s) at the site; site st...


Sample AI prompt preview:

You are a Jio customer support assistant. Draft a short, polite update for a customer. Do not include personal details and do not promise refunds, fixes or resolution dates.
Case reference: CMP000001
Location: Jaipur, Rajasthan
Issue type: Unknown
Current status: Open
Priority: P1 - High
Internal findings: 1 open severe incident(s) at the site; site status Planned Upgrade; customer has 4 complaints
Next step for our team: Investigate the affected network site and update the customer


## Milestone 10 · Deliver operational reports and a searchable case view

**From the team:** The CareOps supervisor needs an exportable complaint queue, circle summary and quick filtering tool that works inside Colab without external APIs.

**Your responsibility:** Sort in P0, P1, P2, P3 order. Export `jio_complaint_priority_report.csv` and `jio_careops_operations_summary.xlsx`, offer an optional widget-based search/filter with console fallback, and write 3–5 data-backed findings.

**Handover evidence:** Create `final_report`, `circle_summary`, `priority_summary`, CSV/XLSX exports and `filter_cases` helper.


In [12]:
# Milestone 10 — Deliver operational reports and a searchable case view
# Task 1: Create final_report with one row per complaint and sort by priority P0, P1, P2, P3.
# Task 2: Build circle_summary by circle and case_priority with case_count and unique_customers.
# Task 3: Export final_report to DATA_DIR/'jio_complaint_priority_report.csv'.
# Task 4: Export priority_summary and circle_summary as Excel worksheets in DATA_DIR/'jio_careops_operations_summary.xlsx'.
# Task 5: Provide callable filter_cases to search/filter the report by priority, circle, city, issue group and IDs.
# Task 6: Display saved files and report previews, and write 3–5 evidence-based handover findings.

# Milestone 10 — Deliver operational reports and a searchable case view

# ---------- 1. Final report: one row per complaint, sorted P0 -> P3 ----------
priority_order = ['P0', 'P1', 'P2', 'P3']
final_report = final_cases_df.copy()
final_report['case_priority'] = pd.Categorical(final_report['case_priority'], categories=priority_order, ordered=True)
final_report = (final_report
                .sort_values(['case_priority', 'priority_score', 'customer_complaint_count', 'complaint_id'],
                             ascending=[True, False, False, True])
                .reset_index(drop=True))
final_report['case_priority'] = final_report['case_priority'].astype(str)

# ---------- 2. Circle summary and action-owner workload ----------
circle_summary = (final_report.groupby(['circle', 'case_priority'])
    .agg(case_count=('complaint_id', 'size'), unique_customers=('customer_id', 'nunique'))
    .reset_index())
circle_summary['case_priority'] = pd.Categorical(circle_summary['case_priority'], categories=priority_order, ordered=True)
circle_summary = circle_summary.sort_values(['circle', 'case_priority']).reset_index(drop=True)
circle_summary['case_priority'] = circle_summary['case_priority'].astype(str)

action_owner_workload = (pd.crosstab(final_report['action_owner'], final_report['case_priority'])
    .reindex(columns=priority_order, fill_value=0))
action_owner_workload['total_cases'] = action_owner_workload.sum(axis=1)
action_owner_workload = action_owner_workload.sort_values('total_cases', ascending=False).reset_index()

# ---------- 3. Export the two reports ----------
csv_path = DATA_DIR / 'jio_complaint_priority_report.csv'
xlsx_path = DATA_DIR / 'jio_careops_operations_summary.xlsx'

csv_export = final_report.copy()
csv_export['ai_prompt_preview'] = csv_export['ai_prompt_preview'].str.replace('\n', ' | ', regex=False)
csv_export.to_csv(csv_path, index=False)

with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    priority_summary.to_excel(writer, sheet_name='priority_summary', index=False)
    circle_summary.to_excel(writer, sheet_name='circle_summary', index=False)
    location_summary.to_excel(writer, sheet_name='location_summary', index=False)
    action_owner_workload.to_excel(writer, sheet_name='action_owner_workload', index=False)

# ---------- 4. Search / filter helper ----------
DISPLAY_COLUMNS = ['complaint_id', 'customer_id', 'city', 'circle', 'issue_group', 'complaint_status',
                   'case_priority', 'priority_score', 'action_owner', 'recommended_action']

def filter_cases(priority=None, circle=None, city=None, issue_group=None,
                 complaint_id=None, customer_id=None, limit=20):
    """Filter final_report. Ignores case and extra spaces; handles invalid input without crashing."""
    result = final_report
    if priority:
        wanted = [p.strip().upper() for p in ([priority] if isinstance(priority, str) else priority)]
        invalid = [p for p in wanted if p not in priority_order]
        if invalid:
            print(f"Invalid priority {invalid}. Choose from {priority_order}.")
            return final_report[DISPLAY_COLUMNS].iloc[0:0]
        result = result[result['case_priority'].isin(wanted)]
    for column, value in [('circle', circle), ('city', city), ('issue_group', issue_group)]:
        if value:
            result = result[result[column].astype('string').str.strip().str.casefold()
                            == str(value).strip().casefold()]
    for column, value in [('complaint_id', complaint_id), ('customer_id', customer_id)]:
        if value:
            result = result[result[column].astype('string').str.strip().str.upper()
                            == str(value).strip().upper()]
    if result.empty:
        print("No complaints match those filters.")
        return final_report[DISPLAY_COLUMNS].iloc[0:0]
    print(f"{len(result)} complaint(s) found; showing {min(limit, len(result))}.")
    return result[DISPLAY_COLUMNS].head(limit)

# ---------- 5. Optional interactive widget (falls back to filter_cases if unavailable) ----------
def build_filter_widget():
    try:
        import ipywidgets as widgets
        from IPython.display import clear_output
    except ImportError:
        print("Widgets not available - call filter_cases(priority='P0', city='Pune') instead.")
        return
    def options(column):
        return ['All'] + sorted(final_report[column].dropna().astype(str).unique())
    priority_dd = widgets.Dropdown(options=['All'] + priority_order, description='Priority:')
    circle_dd = widgets.Dropdown(options=options('circle'), description='Circle:')
    city_dd = widgets.Dropdown(options=options('city'), description='City:')
    group_dd = widgets.Dropdown(options=options('issue_group'), description='Issue:')
    id_text = widgets.Text(description='Complaint ID:', placeholder='e.g. CMP000001')
    search_button = widgets.Button(description='Search', button_style='primary')
    output = widgets.Output()

    def on_search(_):
        pick = lambda w: None if w.value == 'All' else w.value
        with output:
            clear_output()
            display(filter_cases(priority=pick(priority_dd), circle=pick(circle_dd), city=pick(city_dd),
                                 issue_group=pick(group_dd), complaint_id=id_text.value.strip() or None))
    search_button.on_click(on_search)
    display(widgets.VBox([priority_dd, circle_dd, city_dd, group_dd, id_text, search_button, output]))

build_filter_widget()

# ---------- 6. Evidence: saved files and previews ----------
print("Saved files:")
for path in (csv_path, xlsx_path):
    print(f"  {path.name}  ({path.stat().st_size / 1024:.0f} KB)")
print("\nReport rows:", len(final_report), "| unique complaints:", final_report['complaint_id'].is_unique,
      "| first priority:", final_report['case_priority'].iloc[0], "| last priority:", final_report['case_priority'].iloc[-1])
print("Excel sheets:", pd.ExcelFile(xlsx_path).sheet_names)

print("\nPriority summary:");        display(priority_summary)
print("Circle summary (preview):");  display(circle_summary.head(8))
print("Action-owner workload:");     display(action_owner_workload)
print("Report preview:");            display(final_report[DISPLAY_COLUMNS].head(10))

# ---------- 7. Numbers for your 3-5 written findings ----------
unresolved = ~final_report['complaint_status'].isin(['Resolved', 'Closed'])
high = final_report[final_report['case_priority'].isin(['P0', 'P1'])]
print("\n--- Evidence for findings ---")
print("Most complaints by circle:", final_report['circle'].value_counts().head(3).to_dict())
print("Most complaints by city:  ", final_report['city'].value_counts().head(3).to_dict())
print("P0 + P1 complaints:", len(high), f"({len(high) / len(final_report):.1%})",
      "| still unresolved:", (~high['complaint_status'].isin(['Resolved', 'Closed'])).sum())
print("P0 already Resolved/Closed:", ((final_report['case_priority'] == 'P0') & ~unresolved).sum())
print("P0 by circle:", final_report[final_report['case_priority'] == 'P0']['circle'].value_counts().head(3).to_dict())
print("Complaints with unresolved refunds:", final_report['recharge_risk_signal'].sum(),
      "| refund amount tied to them: Rs.", int(final_report['unresolved_refund_amount'].sum()))
print("Complaints with escalated/SLA-review requests:", final_report['service_followup_signal'].sum())
print("Complaints from repeat customers (3+):", final_report['repeat_contact_signal'].sum(),
      f"({final_report['repeat_contact_signal'].mean():.1%})")
print("Data quality: duplicate IDs removed", len(duplicate_rows_removed),
      "| unknown customers", final_report['missing_customer_ref'].sum(),
      "| unknown sites", final_report['missing_site_ref'].sum(),
      "| missing status", final_report['complaint_status'].isna().sum(),
      "| unknown issue group", (final_report['issue_group'] == 'Unknown').sum())

try:
    from google.colab import files
    files.download(str(csv_path)); files.download(str(xlsx_path))
except Exception:
    print("\n(Download the two files from the Colab file panel if the automatic download did not start.)")

Saved files:
  jio_complaint_priority_report.csv  (5184 KB)
  jio_careops_operations_summary.xlsx  (9 KB)

Report rows: 4901 | unique complaints: True | first priority: P0 | last priority: P3
Excel sheets: ['priority_summary', 'circle_summary', 'location_summary', 'action_owner_workload']

Priority summary:


,case_priority,case_count,share_pct
0,P0,584,11.9
1,P1,1888,38.5
2,P2,2224,45.4
3,P3,205,4.2


Circle summary (preview):


,circle,case_priority,case_count,unique_customers
0,Andhra Pradesh & Telangana,P0,62,61
1,Andhra Pradesh & Telangana,P1,147,138
2,Andhra Pradesh & Telangana,P2,158,149
3,Andhra Pradesh & Telangana,P3,14,14
4,Delhi NCR,P0,51,49
5,Delhi NCR,P1,162,151
6,Delhi NCR,P2,212,192
7,Delhi NCR,P3,15,14


Action-owner workload:


case_priority,action_owner,P0,P1,P2,P3,total_cases
0,Customer Care - Repeat Follow-up,0,0,2025,0,2025
1,Network Operations,485,987,110,0,1582
2,Recharge Operations,99,474,41,0,614
3,Customer Care,0,427,48,0,475
4,Standard Support,0,0,0,205,205


... (3 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Handover findings · 3–5 evidence-based observations
Write three to five findings supported by computed numbers, including the most affected circle/city, the actionable Critical/High workload, recharge/service risks, one operational recommendation, and at least one data-quality limitation.

**Your findings:**

1. Most affected locations: Gujarat has the most complaints of any circle (779), but that is two cities combined, Surat and Ahmedabad. The busiest single city is Indore (452 complaints), followed by Pune (447) and Bengaluru (441). Kolkata has the most complaints at sites with an open severe network incident (166).

2. Actionable workload: 584 complaints are P0 and 1,888 are P1, so 2,472 (50.4%) need priority attention, and 1,787 of those are still unresolved. Network Operations owns 485 of the P0 cases and Recharge Operations owns 99. 169 P0 cases are already Resolved or Closed and should be reviewed separately from the live queue.

3. Recharge and service risk: 924 complaints have at least one unresolved refund, tied to about ₹341,041. 868 complaints have an escalated service request or one over the 72-hour SLA, and 840 of the 1,500 requests (56%) are escalated.

4. Recommendation: work the 485 P0 cases owned by Network Operations first, since each has an open severe incident plus at least two other risks. Then clear the 99 P0 refund cases. Because 90.6% of complaints come from customers with three or more complaints, the repeat-contact signal does little to separate cases, and a stricter threshold (for example five or more) should be tested.

5. Data-quality limitations: CMP001502 appeared 100 times with different details, and I kept the first row (99 removed). No site has a Down, Inactive or Outage status, so the network signal rests only on open Critical and Major Outage reports. 452 complaints have no status and 282 have no issue category. The 72-hour SLA is my own assumption, since none is documented. The orphan records are 4 recharges, 3 service requests, and 30 network reports at unknown sites.

## Fix log and edge cases
| Problem found | Fix |
|---|---|
| Assessment flagged missing `open_request_count` | Added it to the Milestone 6 aggregation and the Milestone 8 count columns |
| complaint_id CMP001502 repeated 100 times with different details | Kept the first row, saved the 99 removed rows in `duplicate_rows_removed` |
| Mixed spellings in statuses, severity and categories | Lowercased, trimmed and mapped to standard labels |
| Recharge amounts like "₹149" and "-199" | Stripped symbols, converted to numbers, flagged negatives |
| Invalid dates ("not_a_date", "2026-15-01", "bad-date") | Converted to NaT and flagged, not deleted |
| Orphan records (4 recharges, 3 requests, 30 network reports) | Kept in separate tables for review |
| Invalid filter input (e.g. priority "P9") | `filter_cases` prints a message instead of crashing |

### Final milestone · Automated assessment
**Important:** Colab retains previously created variables and files. After changing or deleting any milestone code, choose **Runtime → Restart session**, then **Run all** before trusting the self-check. Running only the assessment cell can show incorrect passes. For official grading, the instructor must execute your saved `.ipynb` in a fresh kernel with clean input files.

Run the self-check **after all ten milestones**. It creates a scorecard for your notebook using observable DataFrame outputs and files. It does **not** require an answer key to be shared with students.

**How to read results:** `Pass` means the available output meets the named structural or business-rule check; `Review` means something is absent or differs from the expected contract. This is **formative self-checking**, not a secure examination: notebook code can be modified and passing checks do not certify every analysis table or commentary. For official grading, retain a human review of findings, data quality decisions, and a sample of calculations.

**Student instruction:** Run all cells from the top before evaluating. Do not edit the assessment cell.

In [13]:

# AUTOMATED MILESTONE ASSESSMENT — Run only after finishing Milestones 1–10.
# This is a non-secure learning aid; see the project brief for review expectations.

import pandas as pd
from pathlib import Path
from IPython.display import display

_scope = globals()
_checks = []

def check(task, criterion, fn):
    try:
        passed = bool(fn())
        msg = 'Meets check' if passed else 'Missing or inconsistent output'
    except Exception as ex:
        passed = False
        msg = f'Cannot verify: {type(ex).__name__}'
    _checks.append({
        'Milestone': task,
        'Check': criterion,
        'Result': 'Pass' if passed else 'Review',
        'Points': 5 if passed else 0,
        'Feedback': msg
    })

def df(name):
    obj = _scope.get(name)
    if not isinstance(obj, pd.DataFrame):
        raise ValueError(f'{name} is not a DataFrame')
    return obj

def has_cols(name, cols):
    return set(cols).issubset(df(name).columns)

def unique_nonnull(frame, key):
    return key in frame and frame[key].notna().all() and frame[key].is_unique

def numeric(frame, col):
    return col in frame and pd.api.types.is_numeric_dtype(frame[col])

def real_dates(frame, cols):
    return all(
        col in frame and pd.api.types.is_datetime64_any_dtype(frame[col])
        for col in cols
    )

def source_counts_match(summary_name, source_name, key, count_col):
    summary = df(summary_name)
    source = df(source_name)
    expected = source[key].notna().sum()
    return (
        unique_nonnull(summary, key)
        and numeric(summary, count_col)
        and summary[count_col].sum() == expected
    )

def network_counts_match():
    n = df('network_clean')
    s = df('network_by_site')
    if not has_cols('network_by_site', [
        'cell_site_id', 'network_report_count',
        'severe_network_report_count'
    ]):
        return False
    expected = (
        n.dropna(subset=['cell_site_id'])
        .groupby('cell_site_id').size()
    )
    actual = s.set_index('cell_site_id')['network_report_count']
    return (
        unique_nonnull(s, 'cell_site_id')
        and actual.reindex(expected.index).eq(expected).all()
        and actual.sum() == len(n.dropna(subset=['cell_site_id']))
    )

def issues_match():
    cases = df('final_cases_df')
    for source_name, count_col in [
        ('recharge_clean', 'recharge_count'),
        ('requests_clean', 'request_count')
    ]:
        source = df(source_name)
        if count_col not in cases.columns:
            return False
        counts = (
            source[source.complaint_id.isin(cases.complaint_id)]
            .groupby('complaint_id').size()
        )
        actual = cases.set_index('complaint_id')[count_col].fillna(0)
        expected = counts.reindex(actual.index, fill_value=0)
        if not actual.eq(expected).all():
            return False
    return True

def flags_match():
    c = df('final_cases_df')
    flags = [
        'severe_network_signal',
        'recharge_risk_signal',
        'service_followup_signal',
        'repeat_contact_signal'
    ]
    if not has_cols('final_cases_df', flags):
        return False
    if any(not pd.api.types.is_bool_dtype(c[x]) for x in flags):
        return False

    network_expected = (
        c.severe_network_report_count.fillna(0).gt(0)
        | c.site_status.astype('string').str.strip()
          .str.casefold().isin(['down', 'inactive', 'outage'])
    ).fillna(False)

    recharge_expected = c.recharge_risk_count.fillna(0).gt(0)

    service_expected = (
        c.escalated_request_count.fillna(0).gt(0)
        | c.sla_review_count.fillna(0).gt(0)
    )

    repeat_expected = c.customer_complaint_count.fillna(0).ge(3)

    return all(
        c[col].fillna(False).eq(expected).all()
        for col, expected in zip(
            flags,
            [
                network_expected,
                recharge_expected,
                service_expected,
                repeat_expected
            ]
        )
    )

def priorities_match():
    c = df('final_cases_df')
    flags = [
        'severe_network_signal',
        'recharge_risk_signal',
        'service_followup_signal',
        'repeat_contact_signal'
    ]
    if not has_cols('final_cases_df', flags + [
        'priority_score', 'case_priority'
    ]):
        return False
    if any(not pd.api.types.is_bool_dtype(c[x]) for x in flags):
        return False

    signals = c[flags].sum(axis=1)
    expected = signals.map(
        lambda x: 'P0' if x >= 3 else (
            'P1' if x == 2 else ('P2' if x == 1 else 'P3')
        )
    )
    actual = (
        c.case_priority.astype('string').str.strip()
        .str.extract(r'^(P[0-3])', expand=False)
    )
    return (
        c.priority_score.eq(signals).all()
        and actual.eq(expected).all()
    )

def report_matches():
    r = df('final_report')
    c = df('final_cases_df')
    return (
        has_cols('final_report', ['complaint_id', 'case_priority'])
        and len(r) == len(c)
        and unique_nonnull(r, 'complaint_id')
        and set(r.complaint_id) == set(c.complaint_id)
        and r.set_index('complaint_id').case_priority.eq(
            c.set_index('complaint_id')
            .loc[r.complaint_id, 'case_priority'].to_numpy()
        ).all()
    )

def summary_matches():
    s = df('priority_summary')
    if not has_cols('priority_summary', [
        'case_priority', 'case_count'
    ]):
        return False
    expected = df('final_cases_df').case_priority.value_counts()
    got = s.groupby('case_priority').case_count.sum()
    return (
        got.reindex(expected.index, fill_value=0).eq(expected).all()
        and got.sum() == len(df('final_cases_df'))
    )

check(1, 'Complaint source loaded, key fields present',
      lambda: has_cols('complaints_df',
      ['complaint_id', 'customer_id', 'cell_site_id', 'complaint_status']))

check(1, 'Complaint source is non-empty',
      lambda: len(df('complaints_df')) > 0)

check(2, 'Network and recharge sources loaded',
      lambda: has_cols('network_df',
      ['network_report_id', 'cell_site_id', 'severity'])
      and has_cols('recharge_df',
      ['recharge_id', 'complaint_id', 'recharge_status'])
      and len(df('network_df')) > 0
      and len(df('recharge_df')) > 0)

check(2, 'Service request source loaded',
      lambda: has_cols('requests_df',
      ['service_request_id', 'complaint_id', 'service_status'])
      and len(df('requests_df')) > 0)

check(3, 'One clean record per complaint',
      lambda: unique_nonnull(df('complaints_clean'), 'complaint_id')
      and len(df('complaints_clean')) <= len(df('complaints_df')))

check(3, 'Complaint dates and categories standardized',
      lambda: real_dates(df('complaints_clean'), ['complaint_date'])
      and has_cols('complaints_clean',
      ['issue_group', 'complaint_status', 'city', 'circle']))

check(4, 'Operational records cleaned',
      lambda: unique_nonnull(df('network_clean'), 'network_report_id')
      and unique_nonnull(df('recharge_clean'), 'recharge_id')
      and unique_nonnull(df('requests_clean'), 'service_request_id'))

check(4, 'Operational dates standardized',
      lambda: real_dates(df('network_clean'), ['report_date'])
      and real_dates(df('recharge_clean'), ['recharge_date'])
      and real_dates(df('requests_clean'), ['request_created_date']))

check(5, 'Network reports aggregated by site',
      lambda: unique_nonnull(df('network_by_site'), 'cell_site_id')
      and has_cols('network_by_site',
      ['network_report_count', 'severe_network_report_count']))

check(5, 'Network report totals reconcile',
      network_counts_match)

check(6, 'Recharge summary uses unique complaint IDs',
      lambda: unique_nonnull(df('recharge_by_complaint'), 'complaint_id')
      and has_cols('recharge_by_complaint',
      ['recharge_count', 'recharge_risk_count']))

check(6, 'Recharge totals match cleaned source',
      lambda: source_counts_match(
          'recharge_by_complaint', 'recharge_clean',
          'complaint_id', 'recharge_count'
      ))

check(7, 'Service requests summarized by complaint',
      lambda: unique_nonnull(df('requests_by_complaint'), 'complaint_id')
      and has_cols('requests_by_complaint',
      ['request_count', 'open_request_count',
       'escalated_request_count', 'sla_review_count'])
      and source_counts_match(
          'requests_by_complaint', 'requests_clean',
          'complaint_id', 'request_count'
      ))

check(7, 'Repeat customer counts reconcile',
      lambda: has_cols('customer_repeat_count',
      ['customer_id', 'customer_complaint_count'])
      and unique_nonnull(df('customer_repeat_count'), 'customer_id')
      and df('customer_repeat_count').customer_complaint_count.sum()
      == df('complaints_clean').customer_id.notna().sum())

check(8, 'Master table contains all complaints once',
      lambda: unique_nonnull(df('final_cases_df'), 'complaint_id')
      and set(df('final_cases_df').complaint_id)
      == set(df('complaints_clean').complaint_id))

check(8, 'Master issue counts reconcile',
      issues_match)

check(9, 'Operational risk signals follow business rules',
      flags_match)

check(9, 'Priority mapping matches risk signals',
      priorities_match)

check(10, 'Detailed report and export exist',
      lambda: report_matches()
      and (Path(DATA_DIR) / 'jio_complaint_priority_report.csv').is_file())

check(10, 'Priority summary and export exist',
      lambda: summary_matches()
      and (Path(DATA_DIR) / 'jio_careops_operations_summary.xlsx').is_file())

assessment_df = pd.DataFrame(_checks)

score_by_task = assessment_df.groupby(
    'Milestone', as_index=False
).agg(
    Points=('Points', 'sum'),
    Passed=('Result', lambda x: (x == 'Pass').sum())
)

score_by_task['Maximum'] = 10
score_by_task['Status'] = score_by_task['Passed'].map({
    0: 'Review',
    1: 'Partially verified',
    2: 'Checks passed'
})

print('AUTOMATED SELF-ASSESSMENT · NOT FINAL HUMAN GRADE')
display(score_by_task[['Milestone', 'Points', 'Maximum', 'Status']])

print(f'Automated checks score: {assessment_df.Points.sum()}/100')

print('Checks needing attention:')
display(
    assessment_df.loc[
        assessment_df.Result == 'Review',
        ['Milestone', 'Check', 'Feedback']
    ].reset_index(drop=True)
)

# Machine-readable output for independent, fresh-kernel instructor grading.
import json as _grade_json

print(
    'JIO_CAREOPS_ASSESSMENT_JSON='
    + _grade_json.dumps({
        'points': int(assessment_df.Points.sum()),
        'milestones': score_by_task[
            ['Milestone', 'Points', 'Maximum', 'Status']
        ].to_dict('records'),
        'checks': assessment_df[
            ['Milestone', 'Check', 'Result', 'Points']
        ].to_dict('records')
    })
)


AUTOMATED SELF-ASSESSMENT · NOT FINAL HUMAN GRADE


,Milestone,Points,Maximum,Status
0,1,10,10,Checks passed
1,2,10,10,Checks passed
2,3,10,10,Checks passed
3,4,10,10,Checks passed
4,5,10,10,Checks passed
5,6,10,10,Checks passed
6,7,10,10,Checks passed
7,8,10,10,Checks passed
8,9,10,10,Checks passed
9,10,10,10,Checks passed


Automated checks score: 100/100
Checks needing attention:


,Milestone,Check,Feedback


JIO_CAREOPS_ASSESSMENT_JSON={"points": 100, "milestones": [{"Milestone": 1, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 2, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 3, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 4, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 5, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 6, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 7, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 8, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 9, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 10, "Points": 10, "Maximum": 10, "Status": "Checks passed"}], "checks": [{"Milestone": 1, "Check": "Complaint source loaded, key fields present", "Result": "Pass", "Points": 5}, {"Milestone": 1, "Check": "Complaint source is non-empty", "Result": "Pass", "Point

### Submission handover
1. Save your completed Colab copy in Drive and share its link with the required permissions, or submit the `.ipynb` file if instructed by your instructor.
2. Include the final automated scorecard **and** your written findings in the notebook.
3. Do not submit the original three input datasets unless requested; output CSV and Excel files are generated during your run.
